In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os
from tqdm import tqdm

def generate_recent_disturbance_summary_for_year(target_year, disturbance_folder, reference_path, output_path):
    years = list(range(target_year, 1987, -1))  # From target_year down to 1988

    # Read reference raster for shape and metadata
    with rasterio.open(reference_path) as ref:
        profile = ref.profile
        height, width = ref.height, ref.width
        transform = ref.transform
        crs = ref.crs

    # Initialize result arrays
    disturbance_type = np.zeros((height, width), dtype=np.uint8)
    gap_years = np.zeros((height, width), dtype=np.uint8)
    filled_mask = np.zeros((height, width), dtype=bool)

    for year in years:
        path = os.path.join(disturbance_folder, f"Disturbance_{year}_1km_dominant.tif")
        if not os.path.exists(path):
            print(f"Missing: {path}")
            continue

        with rasterio.open(path) as src:
            arr = np.empty((height, width), dtype=np.uint8)
            reproject(
                source=src.read(1),
                destination=arr,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=crs,
                resampling=Resampling.nearest
            )

        valid_mask = (~filled_mask) & (arr >= 1) & (arr <= 8)
        disturbance_type[valid_mask] = arr[valid_mask]
        gap_years[valid_mask] = target_year - year
        filled_mask[valid_mask] = True

    # For pixels with no past disturbance
    no_disturbance_mask = ~filled_mask
    disturbance_type[no_disturbance_mask] = 0
    gap_years[no_disturbance_mask] = 0 if target_year == 1988 else target_year - 1988

    # Update profile and save
    profile.update({
        "count": 2,
        "dtype": "uint8",
        "compress": "lzw"
    })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(disturbance_type, 1)
        dst.write(gap_years, 2)

    print(f"Saved for year {target_year}: {output_path}")


# ===== Loop through each target year from 1988 to 2021 =====
disturbance_folder = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\1km_dominant_forest_disturbance"
reference_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\1km_dominant_forest_disturbance/Disturbance_1985_1km_dominant.tif"
output_root = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\1km_dominant_forest_disturbance\Recent_Summaries"

os.makedirs(output_root, exist_ok=True)

for year in tqdm(range(1988, 2022)):
    output_path = os.path.join(output_root, f"Most_Recent_Disturbance_Summary_{year}.tif")
    generate_recent_disturbance_summary_for_year(year, disturbance_folder, reference_path, output_path)